# Notebook 15 - Topic Classification: Pipeline vs LLM Comparison

## Two-stage methodology
**Stage 1 (Notebook 06b):** BERTopic discovers the topic taxonomy from 1115 articles.
BERTopic is a corpus analysis tool - it does NOT participate in classification or gold standard construction.

**Stage 2 (this notebook):** Three NLI pipelines and four LLMs independently classify
183 validation articles. Dual gold standard mirrors Notebook 14 exactly.

## Taxonomy (6 categories + OTHER)
| Category | Core concept |
|---|---|
| Energy Efficiency and Renovation | Heat pumps, insulation, energy renovation |
| Housing Market Affordability and Financing | Rents, housing shortage, KfW grants |
| Sustainable Materials and Construction | Timber, recycled materials, circular building |
| Architecture Planning and Urban Development | Architects, regulations, urban renewal |
| Climate and Energy Policy | Renewable energy, EU policy, energy transition |
| Residential Building Types | Tiny houses, prefab, campus, community housing |
| OTHER | Off-topic or unclassifiable |


In [ ]:
# CELL 1 : INSTALLATION
!pip install -q groq transformers torch scikit-learn
!pip install -q "numpy>=2.0"


In [ ]:
# CELL 2 : IMPORTS & CONFIGURATION
import os, json, pickle, time, re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

ROOT        = Path("/content/drive/MyDrive/thesis")
DATA_PROC   = ROOT / "Project/Data/Processed"
FIGURES_DIR = ROOT / "Project/Outputs/Figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

GROQ_TOKEN = userdata.get("GROQ_TOKEN")

CATEGORIES = [
    "Energy Efficiency and Renovation",
    "Housing Market Affordability and Financing",
    "Sustainable Materials and Construction",
    "Architecture Planning and Urban Development",
    "Climate and Energy Policy",
    "Residential Building Types",
    "OTHER",
]

CATEGORY_DESCRIPTIONS = {
    "Energy Efficiency and Renovation":
        "heat pumps, thermal insulation, energy renovation of buildings, "        "reducing CO2 in existing buildings, energy certificates, window replacement",
    "Housing Market Affordability and Financing":
        "rent prices, housing shortage, affordable housing, real estate market, "        "KfW subsidies, government housing grants, mortgage rates, housing programs",
    "Sustainable Materials and Construction":
        "timber construction, wood as building material, recycled building materials, "        "concrete recycling, circular construction, sustainable interior design",
    "Architecture Planning and Urban Development":
        "architects, building design, urban planning, building regulations, "        "demolition and redevelopment, historic buildings, construction permits",
    "Climate and Energy Policy":
        "renewable energy policy, government climate action plans, EU energy regulations, "        "energy transition, hydrogen economy, climate targets, political energy decisions",
    "Residential Building Types":
        "tiny houses, prefabricated homes, single-family houses, campus buildings, "        "community housing projects, self-build housing, student accommodation",
    "OTHER":
        "articles that do not fit any of the above categories, "        "or are about unrelated topics such as cars, agriculture, or tourism",
}

TOPIC_TO_CATEGORY = {
    16: "Energy Efficiency and Renovation",
    9 : "Housing Market Affordability and Financing",
    10: "Housing Market Affordability and Financing",
    14: "Housing Market Affordability and Financing",
    11: "Sustainable Materials and Construction",
    15: "Sustainable Materials and Construction",
    22: "Sustainable Materials and Construction",
    7 : "Architecture Planning and Urban Development",
    21: "Architecture Planning and Urban Development",
    23: "Architecture Planning and Urban Development",
    24: "Architecture Planning and Urban Development",
    4 : "Climate and Energy Policy",
    17: "Climate and Energy Policy",
    18: "Climate and Energy Policy",
    5 : "Residential Building Types",
    6 : "Residential Building Types",
    13: "Residential Building Types",
}

NLI_MODELS = {
    "mDeBERTa-NLI"      : "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "XLM-RoBERTa-NLI"   : "joeddav/xlm-roberta-large-xnli",
    "DeBERTa-TaskSource" : "MoritzLaurer/deberta-v3-base-tasksource-nli",
}

LLM_MODELS = {
    "llama-3.1-8b-instant"                     : {"scale_B": 8,  "family": "llama"},
    "meta-llama/llama-4-scout-17b-16e-instruct": {"scale_B": 17, "family": "llama4"},
    "qwen/qwen3-32b"                           : {"scale_B": 32, "family": "qwen3"},
    "llama-3.3-70b-versatile"                  : {"scale_B": 70, "family": "llama"},
}

LLM_GOLD_MODELS = [
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "qwen/qwen3-32b",
    "llama-3.3-70b-versatile",
]

SAVE_EVERY               = 10
MAX_CONSECUTIVE_FAILURES = 5
RETRY_DELAYS             = [30, 60, 120]
MODEL_SLEEP = {
    "llama-3.3-70b-versatile"                  : 35.0,
    "meta-llama/llama-4-scout-17b-16e-instruct": 4.0,
    "qwen/qwen3-32b"                           : 3.0,
    "llama-3.1-8b-instant"                     : 2.0,
}

print("Configuration loaded")
print(f"  Categories : {len(CATEGORIES)}")
print(f"  NLI models : {list(NLI_MODELS.keys())}")
print(f"  LLM models : {list(LLM_MODELS.keys())}")


In [ ]:
# CELL 3 : LOAD DATA
print("Loading data...")
pipeline_df = pd.read_pickle(DATA_PROC / "ner_pipeline_results.pkl")
assignments = pd.read_csv(DATA_PROC / "topic_assignments_v2.csv")

with open(DATA_PROC / "ner_llm_checkpoint.pkl", "rb") as f:
    qwen_ckpt = pickle.load(f)
VAL_IDS = set(aid for aid, data in qwen_ckpt.items()
              if isinstance(data, dict) and data.get("status") != "api_error")
print(f"  Validation set  : {len(VAL_IDS)} articles")

val_df = pipeline_df[pipeline_df["article_id"].isin(VAL_IDS)].copy().reset_index(drop=True)
print(f"  Found in corpus : {len(val_df)} articles")

art_to_topic = dict(zip(assignments["article_id"], assignments["topic"]))
val_df["bertopic_topic"]    = val_df["article_id"].map(art_to_topic)
val_df["bertopic_category"] = val_df["bertopic_topic"].map(TOPIC_TO_CATEGORY).fillna("OTHER")

id_to_de = dict(zip(val_df["article_id"], val_df["content"]))
id_to_en = dict(zip(val_df["article_id"], val_df["content_en"]))
ARTICLE_IDS = val_df["article_id"].tolist()

print(f"
  BERTopic category distribution (reference only):")
print(val_df["bertopic_category"].value_counts().to_string())


In [ ]:
# CELL 4 : NLI PIPELINE CLASSIFICATION
# Three NLI models classify each article into the taxonomy.
# Input: German text (content) - pipelines work in source language.
# Checkpointed - re-running is safe.

from transformers import pipeline as hf_pipeline
import gc, torch

def ckpt_path_nli(model_name):
    safe = re.sub(r"[/\:.]", "_", model_name)
    return DATA_PROC / f"nb15_nli_{safe}_checkpoint.pkl"

def load_nli_ckpt(model_name):
    fp = ckpt_path_nli(model_name)
    if fp.exists():
        with open(fp, "rb") as f: ckpt = pickle.load(f)
        done = len([r for r in ckpt["results"] if r["status"] == "ok"])
        print(f"  Resuming {model_name}: {done} done")
        return ckpt
    return {"model": model_name, "results": [], "failed_ids": [], "stop_reason": None}

def save_nli_ckpt(ckpt, model_name):
    with open(ckpt_path_nli(model_name), "wb") as f: pickle.dump(ckpt, f)

nli_results = {}

for model_name, model_id in NLI_MODELS.items():
    ckpt = load_nli_ckpt(model_name)
    if ckpt.get("stop_reason") == "complete":
        n_ok = len([r for r in ckpt["results"] if r["status"] == "ok"])
        print(f"  {model_name} already complete ({n_ok} articles)")
        nli_results[model_name] = ckpt
        continue

    done_ids  = {r["article_id"] for r in ckpt["results"]} | set(ckpt["failed_ids"])
    remaining = [aid for aid in ARTICLE_IDS if aid not in done_ids]

    print(f"
{chr(61)*60}")
    print(f"  NLI MODEL : {model_name}")
    print(f"  Pending   : {len(remaining)} articles")

    try:
        classifier = hf_pipeline(
            "zero-shot-classification",
            model  = model_id,
            device = 0 if torch.cuda.is_available() else -1,
        )
        print(f"  Model loaded")
    except Exception as e:
        print(f"  Failed to load {model_name}: {e}")
        continue

    consec_fail = 0
    for i, aid in enumerate(remaining, start=1):
        text = id_to_de.get(aid, "")
        if not text:
            ckpt["failed_ids"].append(aid); consec_fail += 1; continue
        try:
            result = classifier(
                text[:2000],
                candidate_labels    = CATEGORIES,
                hypothesis_template = "Dieser Artikel handelt von {}.",
                multi_label         = False,
            )
            predicted = result["labels"][0]
            ckpt["results"].append({
                "article_id": aid,
                "predicted" : predicted,
                "scores"    : dict(zip(result["labels"], result["scores"])),
                "status"    : "ok",
            })
            consec_fail = 0
            if i % 20 == 0 or i == len(remaining):
                print(f"  [{i}/{len(remaining)}] last={predicted[:35]}")
        except Exception as e:
            ckpt["failed_ids"].append(aid); consec_fail += 1
            if consec_fail >= MAX_CONSECUTIVE_FAILURES:
                ckpt["stop_reason"] = "consecutive_failures"
                save_nli_ckpt(ckpt, model_name)
                print(f"  STOPPING - {consec_fail} consecutive failures")
                break
        if i % SAVE_EVERY == 0:
            save_nli_ckpt(ckpt, model_name)

    ckpt["stop_reason"] = "complete"
    save_nli_ckpt(ckpt, model_name)
    n_ok = len([r for r in ckpt["results"] if r["status"] == "ok"])
    print(f"  COMPLETE: {model_name} - {n_ok} articles")
    nli_results[model_name] = ckpt

    del classifier
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("
All NLI runs complete")
for name, ckpt in nli_results.items():
    n_ok = len([r for r in ckpt["results"] if r["status"] == "ok"])
    print(f"  {name}: {n_ok} articles [{ckpt.get('stop_reason','?')}]")


In [ ]:
# CELL 5 : LLM CLASSIFICATION PROMPTS

def build_category_list():
    lines = []
    for i, cat in enumerate(CATEGORIES, 1):
        desc = CATEGORY_DESCRIPTIONS.get(cat, "")
        lines.append(f"  {i}. {cat}
     ({desc})")
    return "
".join(lines)

CATEGORY_LIST_STR = build_category_list()

LLM_SYSTEM_PROMPT = (
    "You are a text classification system for sustainable building discourse.
"
    "Classify the given article into exactly ONE of the following categories.
"
    "Return ONLY the category name - no explanation, no numbering, no extra text.

"
    "Categories:
" + CATEGORY_LIST_STR +
    "

If the article does not fit any category, return: OTHER"
)

LLM_USER_TEMPLATE = "Classify this article:

{text}"

def build_llm_messages(model_id, text, condition):
    user_content = LLM_USER_TEMPLATE.format(text=text[:6000])
    if "qwen3" in model_id.lower():
        messages   = [{"role": "system", "content": LLM_SYSTEM_PROMPT},
                      {"role": "user",   "content": "/no_think

" + user_content}]
        max_tokens = 50
    else:
        messages   = [{"role": "system", "content": LLM_SYSTEM_PROMPT},
                      {"role": "user",   "content": user_content}]
        max_tokens = 50
    return messages, max_tokens

def parse_llm_category(raw):
    if not raw: return "OTHER"
    raw = raw.strip()
    if "<think>" in raw:
        if "</think>" in raw:
            raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        else:
            raw = ""
    raw_lower = raw.lower().strip()
    for cat in CATEGORIES:
        if cat.lower() == raw_lower: return cat
    for cat in CATEGORIES:
        if cat.lower() in raw_lower or raw_lower in cat.lower(): return cat
    kw_map = {
        "Energy Efficiency and Renovation"            : ["energy effici","renovation","sanierung","waermepumpe"],
        "Housing Market Affordability and Financing"  : ["housing market","affordab","financing","miete","foerder"],
        "Sustainable Materials and Construction"      : ["sustainable mater","timber","holz","recycl"],
        "Architecture Planning and Urban Development" : ["archite","planning","urban","stadtentwicklung"],
        "Climate and Energy Policy"                   : ["climate","energy policy","klima","energiepolitik"],
        "Residential Building Types"                  : ["residential","tiny house","fertighaus","einfamilien"],
    }
    for cat, kws in kw_map.items():
        if any(kw in raw_lower for kw in kws): return cat
    return "OTHER"

print("LLM prompts ready")
print("
System prompt preview:")
print(LLM_SYSTEM_PROMPT[:500] + "...")


In [ ]:
# CELL 6 : LLM CLASSIFICATION RUNS
# 4 LLMs x 2 conditions (EN + DE) = 8 runs. Checkpointed.

from groq import Groq
groq_client = Groq(api_key=GROQ_TOKEN)

def ckpt_path_llm(model_id, condition):
    safe = re.sub(r"[/\:.-]", "_", model_id)
    return DATA_PROC / f"nb15_llm_{safe}_{condition}_checkpoint.pkl"

def load_llm_ckpt(model_id, condition):
    fp = ckpt_path_llm(model_id, condition)
    if fp.exists():
        with open(fp, "rb") as f: ckpt = pickle.load(f)
        done = len([r for r in ckpt["results"] if r["status"] == "ok"])
        print(f"  Resuming {model_id} [{condition}]: {done} done")
        return ckpt
    return {"model_id": model_id, "condition": condition,
            "results": [], "failed_ids": [], "stop_reason": None}

def save_llm_ckpt(ckpt, model_id, condition):
    with open(ckpt_path_llm(model_id, condition), "wb") as f: pickle.dump(ckpt, f)

def run_llm(model_id, condition, article_ids, id_to_text, sleep_time):
    from groq import RateLimitError, APITimeoutError, APIStatusError
    ckpt = load_llm_ckpt(model_id, condition)
    if ckpt.get("stop_reason") == "complete":
        n_ok = len([r for r in ckpt["results"] if r["status"] == "ok"])
        print(f"  {model_id} [{condition}] complete ({n_ok})")
        return ckpt

    done_ids  = {r["article_id"] for r in ckpt["results"]} | set(ckpt["failed_ids"])
    remaining = [aid for aid in article_ids if aid not in done_ids]

    print(f"
{chr(61)*64}")
    print(f"  MODEL     : {model_id}")
    print(f"  CONDITION : {condition.upper()}")
    print(f"  Pending   : {len(remaining)}")

    consec_fail = 0
    total_ok    = len([r for r in ckpt["results"] if r["status"] == "ok"])

    for i, aid in enumerate(remaining, start=1):
        text = id_to_text.get(aid, "")
        if not text:
            ckpt["failed_ids"].append(aid); consec_fail += 1; continue

        messages, max_tokens = build_llm_messages(model_id, text, condition)
        success = False

        for attempt, delay in enumerate([0] + RETRY_DELAYS, start=1):
            if delay:
                print(f"    Waiting {delay}s (retry {attempt})...")
                time.sleep(delay)
            try:
                resp = groq_client.chat.completions.create(
                    model=model_id, messages=messages,
                    max_tokens=max_tokens, temperature=0.0,
                )
                raw       = resp.choices[0].message.content or ""
                predicted = parse_llm_category(raw)
                ckpt["results"].append({
                    "article_id": aid, "predicted": predicted,
                    "raw": raw, "status": "ok",
                })
                consec_fail = 0; total_ok += 1; success = True
                if i % 20 == 0 or i == len(remaining):
                    print(f"  [{i}/{len(remaining)}] ok={total_ok} last={predicted[:30]}")
                time.sleep(sleep_time)
                break
            except RateLimitError as e:
                if attempt <= len(RETRY_DELAYS):
                    print(f"    429 rate-limit (attempt {attempt}), backing off...")
                else:
                    ckpt["failed_ids"].append(aid); consec_fail += 1
                    ckpt["stop_reason"] = "rate_limit"
                    save_llm_ckpt(ckpt, model_id, condition)
                    print(f"  STOPPING - rate limit. Re-run to resume.")
                    return ckpt
            except (APIStatusError, Exception) as e:
                ckpt["failed_ids"].append(aid); consec_fail += 1
                print(f"  [{i}] {aid} failed: {str(e)[:80]}")
                break

        if not success and aid not in ckpt["failed_ids"]:
            ckpt["failed_ids"].append(aid)

        if consec_fail >= MAX_CONSECUTIVE_FAILURES:
            ckpt["stop_reason"] = "consecutive_failures"
            save_llm_ckpt(ckpt, model_id, condition)
            print(f"  STOPPING - {consec_fail} consecutive failures")
            return ckpt

        if i % SAVE_EVERY == 0:
            save_llm_ckpt(ckpt, model_id, condition)

    ckpt["stop_reason"] = "complete"
    save_llm_ckpt(ckpt, model_id, condition)
    print(f"  COMPLETE: {model_id} [{condition}] ok={total_ok}")
    return ckpt

llm_results = {}
for model_id in LLM_MODELS:
    for condition in ["en", "de"]:
        id_to_text = id_to_en if condition == "en" else id_to_de
        sleep_time = MODEL_SLEEP.get(model_id, 2.0)
        llm_results[(model_id, condition)] = run_llm(
            model_id, condition, ARTICLE_IDS, id_to_text, sleep_time)

print("
LLM RUN SUMMARY")
for (mid, cond), ckpt in llm_results.items():
    n_ok = len([r for r in ckpt["results"] if r["status"] == "ok"])
    n_fail = len(ckpt["failed_ids"])
    print(f"  {mid[-22:]} [{cond}] ok={n_ok} fail={n_fail} [{ckpt.get('stop_reason','?')}]")


In [ ]:
# CELL 7 : BUILD LOOKUPS + SHARED ARTICLE SET

def build_lookup(ckpt):
    return {r["article_id"]: r["predicted"]
            for r in ckpt["results"] if r["status"] == "ok"}

nli_lookups = {name: build_lookup(ckpt) for name, ckpt in nli_results.items()}
llm_lookups_en = {mid: build_lookup(ckpt)
                  for (mid, cond), ckpt in llm_results.items() if cond == "en"}
llm_lookups_de = {mid: build_lookup(ckpt)
                  for (mid, cond), ckpt in llm_results.items() if cond == "de"}

print("Lookups built:")
for name, lk in nli_lookups.items():
    print(f"  NLI {name}: {len(lk)} articles")
for mid, lk in llm_lookups_de.items():
    print(f"  LLM {mid[-22:]} DE: {len(lk)} articles")
for mid, lk in llm_lookups_en.items():
    print(f"  LLM {mid[-22:]} EN: {len(lk)} articles")

# Shared article set - intersection of all complete runs
all_sets = ([set(v.keys()) for v in nli_lookups.values()] +
            [set(v.keys()) for v in llm_lookups_de.values() if len(v) >= 10])
SHARED_IDS = sorted(set.intersection(*all_sets)) if all_sets else []
print(f"
Shared article set: {len(SHARED_IDS)} articles")


In [ ]:
# CELL 8 : BUILD GOLD STANDARDS
# Pipeline Gold: full agreement of 3 NLI pipelines
# LLM Gold DE:   full agreement of top-3 LLMs (DE input)
# Mirrors NB14 dual gold standard design exactly.

from sklearn.metrics import cohen_kappa_score

def full_agreement_gold(lookups_list, article_ids):
    gold = {}
    for aid in article_ids:
        labels = [lk.get(aid) for lk in lookups_list if lk.get(aid)]
        if len(labels) == len(lookups_list) and len(set(labels)) == 1:
            gold[aid] = labels[0]
        else:
            gold[aid] = None
    return gold

def compute_kappa(lk_a, lk_b, article_ids):
    pairs = [(lk_a[aid], lk_b[aid]) for aid in article_ids
             if lk_a.get(aid) and lk_b.get(aid)]
    if len(pairs) < 5: return None
    a, b = zip(*pairs)
    try: return round(cohen_kappa_score(a, b), 3)
    except: return None

nli_list    = [nli_lookups[n] for n in NLI_MODELS if n in nli_lookups]
llm_de_list = [llm_lookups_de[m] for m in LLM_GOLD_MODELS if m in llm_lookups_de]

pipeline_gold_full = full_agreement_gold(nli_list, SHARED_IDS)
llm_gold_full      = full_agreement_gold(llm_de_list, SHARED_IDS)

pg_agreed = {aid: cat for aid, cat in pipeline_gold_full.items() if cat}
lg_agreed = {aid: cat for aid, cat in llm_gold_full.items() if cat}

print(f"Pipeline Gold: {len(pg_agreed)}/{len(SHARED_IDS)} articles agreed")
if pg_agreed:
    print(pd.Series(list(pg_agreed.values())).value_counts().to_string())

print(f"
LLM Gold DE: {len(lg_agreed)}/{len(SHARED_IDS)} articles agreed")
if lg_agreed:
    print(pd.Series(list(lg_agreed.values())).value_counts().to_string())

# Inter-system kappa
print("
Inter-system kappa (Cohen):" )
nli_names = list(NLI_MODELS.keys())
for i in range(len(nli_names)):
    for j in range(i+1, len(nli_names)):
        n1, n2 = nli_names[i], nli_names[j]
        k = compute_kappa(nli_lookups.get(n1,{}), nli_lookups.get(n2,{}), SHARED_IDS)
        print(f"  NLI: {n1} vs {n2}: k={k}")

llm_names = [m for m in LLM_GOLD_MODELS if m in llm_lookups_de]
for i in range(len(llm_names)):
    for j in range(i+1, len(llm_names)):
        m1, m2 = llm_names[i], llm_names[j]
        k = compute_kappa(llm_lookups_de.get(m1,{}), llm_lookups_de.get(m2,{}), SHARED_IDS)
        print(f"  LLM: {m1[-20:]} vs {m2[-20:]}: k={k}")


In [ ]:
# CELL 9 : EVALUATE ALL SYSTEMS
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

def evaluate(predicted_lk, gold_dict, article_ids):
    pairs = [(predicted_lk.get(aid), gold_dict.get(aid))
             for aid in article_ids
             if predicted_lk.get(aid) and gold_dict.get(aid)]
    if len(pairs) < 5:
        return {"n": len(pairs), "accuracy": None, "kappa": None, "macro_f1": None, "per_category": {}}
    pred, gold = zip(*pairs)
    acc  = round(accuracy_score(gold, pred), 4)
    try: kap = round(cohen_kappa_score(gold, pred), 4)
    except: kap = None
    mf1  = round(f1_score(gold, pred, average="macro", zero_division=0), 4)
    per_cat = {}
    for cat in CATEGORIES:
        cp = [1 if p == cat else 0 for p in pred]
        cg = [1 if g == cat else 0 for g in gold]
        if sum(cg) > 0:
            per_cat[cat] = round(f1_score(cg, cp, zero_division=0), 4)
    return {"n": len(pairs), "accuracy": acc, "kappa": kap, "macro_f1": mf1, "per_category": per_cat}

all_systems = {}
for name in NLI_MODELS:
    if name in nli_lookups: all_systems[f"{name} (pipeline)"] = nli_lookups[name]
for mid in LLM_MODELS:
    if mid in llm_lookups_de:
        all_systems[f"{mid.split('/')[-1][:22]} (LLM DE)"] = llm_lookups_de[mid]
for mid in LLM_MODELS:
    if mid in llm_lookups_en:
        all_systems[f"{mid.split('/')[-1][:22]} (LLM EN)"] = llm_lookups_en[mid]

GOLD_STANDARDS = {"Pipeline Gold": pg_agreed, "LLM Gold DE": lg_agreed}

results = {}
for sys_name, sys_lk in all_systems.items():
    results[sys_name] = {gn: evaluate(sys_lk, gd, SHARED_IDS)
                         for gn, gd in GOLD_STANDARDS.items()}

print("Evaluation complete")
print(f"
{chr(61)*75}")
print("RESULTS: Accuracy and kappa - Pipeline Gold vs LLM Gold")
print("(* = EN input - language gap lower bound)")
print(f"{chr(61)*75}")
print(f"  {'System':<45} {'PG Acc':>7} {'PG k':>6} {'LG Acc':>7} {'LG k':>6}")
print(f"  {'-'*72}")
for sys_name in sorted(results, key=lambda s: -(results[s].get("Pipeline Gold",{}).get("accuracy") or 0)):
    pg = results[sys_name].get("Pipeline Gold", {})
    lg = results[sys_name].get("LLM Gold DE", {})
    note = " *" if "(LLM EN)" in sys_name else ""
    print(f"  {sys_name+note:<45} "
          f"{(pg.get('accuracy') or 0):>7.3f} "
          f"{(pg.get('kappa') or 0):>6.3f} "
          f"{(lg.get('accuracy') or 0):>7.3f} "
          f"{(lg.get('kappa') or 0):>6.3f}")


In [ ]:
# CELL 10 : PLOTS
C_PIPELINE = "#1f77b4"
C_LLM_DE   = "#2ca02c"
C_LLM_EN   = "#ff7f0e"

def bar_color(sys_name):
    if "pipeline" in sys_name: return C_PIPELINE
    if "LLM DE"   in sys_name: return C_LLM_DE
    return C_LLM_EN

sys_names = list(all_systems.keys())
x     = np.arange(len(sys_names))
width = 0.38

# Figure 1: Accuracy
fig1, ax1 = plt.subplots(figsize=(13, 7))
pg_accs = [results[s].get("Pipeline Gold",{}).get("accuracy") or 0 for s in sys_names]
lg_accs = [results[s].get("LLM Gold DE",{}).get("accuracy") or 0 for s in sys_names]

b1 = ax1.bar(x - width/2, pg_accs, width, label="Vs Pipeline Gold",
             color=[bar_color(s) for s in sys_names],
             edgecolor="black", linewidth=0.5, alpha=0.9)
b2 = ax1.bar(x + width/2, lg_accs, width, label="Vs LLM Gold DE",
             color=[bar_color(s) for s in sys_names],
             edgecolor="black", linewidth=0.5, alpha=0.5, hatch="//")

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0.01:
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.006,
                 f"{h:.2f}", ha="center", va="bottom", fontsize=7, rotation=90)

short_labels = [s.replace(" (pipeline)","").replace(" (LLM DE)","
(DE)").replace(" (LLM EN)","
(EN)*")
                for s in sys_names]
ax1.set_xticks(x)
ax1.set_xticklabels(short_labels, fontsize=8, rotation=30, ha="right")
ax1.set_ylabel("Accuracy", fontsize=11)
ax1.set_title(f"Topic Classification Accuracy: Pipeline Gold vs LLM Gold
({len(SHARED_IDS)} articles)", fontsize=11)
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1.1)
ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
ax1.grid(axis="y", alpha=0.3)
plt.tight_layout()
for ext in ["pdf","png"]:
    fig1.savefig(FIGURES_DIR/f"nb15_accuracy_dual_gold.{ext}",
                 dpi=300 if ext=="pdf" else 150, bbox_inches="tight")
plt.show()
print("Figure 1 saved: nb15_accuracy_dual_gold")

# Figure 2: Cohen kappa rankings
fig2, axes2 = plt.subplots(1, 2, figsize=(14,6))
for ax2, (gold_name, gcol) in zip(axes2, [("Pipeline Gold", C_PIPELINE),("LLM Gold DE", C_LLM_DE)]):
    ranked = sorted([(s, results[s].get(gold_name,{}).get("kappa") or 0) for s in sys_names], key=lambda t: t[1])
    names_r = [t[0].replace(" (pipeline)","").replace(" (LLM DE)"," DE").replace(" (LLM EN)"," EN*") for t in ranked]
    vals_r  = [t[1] for t in ranked]
    colors_r= [bar_color(t[0]) for t in ranked]
    ax2.barh(names_r, vals_r, color=colors_r, edgecolor="black", linewidth=0.5, alpha=0.85)
    for i, v in enumerate(vals_r):
        ax2.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
    ax2.axvline(x=0.4, color="orange", linewidth=0.8, linestyle="--", alpha=0.5, label="Moderate (0.4)")
    ax2.set_xlabel("Cohen kappa", fontsize=10)
    ax2.set_title(f"Rankings vs {gold_name}", fontsize=11)
    ax2.legend(fontsize=8); ax2.grid(axis="x", alpha=0.3); ax2.tick_params(axis="y", labelsize=8)
fig2.suptitle("Topic Classification: System Rankings by Cohen kappa", fontsize=12, fontweight="bold")
plt.tight_layout()
for ext in ["pdf","png"]:
    fig2.savefig(FIGURES_DIR/f"nb15_kappa_rankings.{ext}",
                 dpi=300 if ext=="pdf" else 150, bbox_inches="tight")
plt.show()
print("Figure 2 saved: nb15_kappa_rankings")


In [ ]:
# CELL 11 : SAVE SUMMARY + FINAL PRINT
nli_kappas, llm_kappas = [], []
nli_list_k    = [nli_lookups[n] for n in NLI_MODELS if n in nli_lookups]
llm_de_list_k = [llm_lookups_de[m] for m in LLM_GOLD_MODELS if m in llm_lookups_de]
for i in range(len(nli_list_k)):
    for j in range(i+1, len(nli_list_k)):
        k = compute_kappa(nli_list_k[i], nli_list_k[j], SHARED_IDS)
        if k: nli_kappas.append(k)
for i in range(len(llm_de_list_k)):
    for j in range(i+1, len(llm_de_list_k)):
        k = compute_kappa(llm_de_list_k[i], llm_de_list_k[j], SHARED_IDS)
        if k: llm_kappas.append(k)

summary = {
    "notebook"          : "15_topic_classification",
    "generated_at"      : datetime.now().isoformat(),
    "n_articles"        : len(SHARED_IDS),
    "categories"        : CATEGORIES,
    "inter_system_kappa": {
        "nli_pipelines_mean": round(float(np.mean(nli_kappas)), 3) if nli_kappas else None,
        "llm_de_mean"       : round(float(np.mean(llm_kappas)), 3) if llm_kappas else None,
    },
    "gold_standards": {
        "pipeline_gold": {"n_agreed": len(pg_agreed)},
        "llm_gold_de"  : {"n_agreed": len(lg_agreed)},
    },
    "results": {
        sys_name: {
            gold_name: {"n": v["n"], "accuracy": v["accuracy"],
                        "kappa": v["kappa"], "macro_f1": v["macro_f1"]}
            for gold_name, v in gold_results.items()
        }
        for sys_name, gold_results in results.items()
    },
}
with open(DATA_PROC / "nb15_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(chr(61)*70)
print("NOTEBOOK 15 COMPLETE")
print(chr(61)*70)
print(f"  Articles      : {len(SHARED_IDS)}")
print(f"  Pipeline Gold : {len(pg_agreed)} agreed")
print(f"  LLM Gold      : {len(lg_agreed)} agreed")
print(f"  NLI kappa mean: {summary['inter_system_kappa']['nli_pipelines_mean']}")
print(f"  LLM kappa mean: {summary['inter_system_kappa']['llm_de_mean']}")
print()
print(f"  {'System':<45} {'PG Acc':>7} {'PG k':>6} {'LG Acc':>7} {'LG k':>6}")
print(f"  {'-'*70}")
for sys_name in sorted(results, key=lambda s: -(results[s].get("Pipeline Gold",{}).get("accuracy") or 0)):
    pg = results[sys_name].get("Pipeline Gold", {})
    lg = results[sys_name].get("LLM Gold DE", {})
    note = " *" if "(LLM EN)" in sys_name else ""
    print(f"  {sys_name+note:<45} {(pg.get('accuracy') or 0):>7.3f} {(pg.get('kappa') or 0):>6.3f} {(lg.get('accuracy') or 0):>7.3f} {(lg.get('kappa') or 0):>6.3f}")
print()
print("  Outputs: nb15_accuracy_dual_gold.pdf/.png")
print("           nb15_kappa_rankings.pdf/.png")
print("           nb15_summary.json")
